# 第7章 関数解析の基礎と再生核ヒルベルト空間 ― デモノートブック

講義ノート第7章は、内積を二変数関数 $k(x,y)$ に取り替えたときに何が起こるかを問い、
**正定値性**・**再生核ヒルベルト空間（RKHS）**・**有限次元と同じ幾何**という三つの答えを与えた。
理論の章なので、このノートブックでは抽象的な対象を数値と図で目に見える形にする。
Gram 行列の固有値、カーネルの断面、$\sum_i\alpha_i k(\cdot,x_i)$ の描画、Mercer 固有関数、
そして分布どうしの距離（MMD）まで、すべて手元で確かめられる。

## 目次

1. [7.1 特徴写像とカーネルトリック](#sec71)（§7.1）
2. [7.2 正定値カーネルと Gram 行列](#sec72)（§7.6、定義 7.x「正定値カーネル」）
3. [7.3 カーネルの演算則](#sec73)（§7.6、定理「カーネルの演算則」・Schur 積定理）
4. [7.4 ガウスカーネルの明示的特徴写像](#sec74)（§7.8、命題「1 次元ガウスカーネルの明示的特徴写像」）
5. [7.5 再生性と RKHS の元](#sec75)（§7.7、定理「再生核の存在」・Moore–Aronszajn）
6. [7.6 Mercer 展開](#sec76)（§7.9、Mercer の定理）
7. [7.7 MMD による二標本の比較](#sec77)（第8章 §8.6 の先取り）
8. [演習](#ex) / [演習の解答](#sol)

> 講義ノートの定理・命題には番号を振っているが、章内の連番は版によって前後するため、
> 以下では**節番号と定理名**で引く。

## 準備

最初にこのセルを実行する。日本語フォントの設定（Colab には既定で入っていない）と、
以降で使うライブラリの読み込みを行う。フォントの導入に失敗した場合は
図のラベルが自動的に英語に切り替わる（`L()` 関数）。

In [ ]:
import subprocess, sys, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore", category=UserWarning)


def _setup_japanese_font():
    """日本語が出せるフォントを探し、なければ入れる。成功したら True。"""
    cands = ["IPAexGothic", "IPAGothic", "Noto Sans CJK JP", "Noto Sans JP",
             "TakaoGothic", "Yu Gothic", "Hiragino Sans"]
    have = {f.name for f in fm.fontManager.ttflist}
    for name in cands:
        if name in have:
            matplotlib.rcParams["font.family"] = name
            return True
    # Colab 想定：pip で導入する
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "japanize-matplotlib"], check=True, timeout=180)
        import japanize_matplotlib  # noqa: F401  読み込むだけで設定される
        return True
    except Exception:
        pass
    # 予備：apt で IPA フォント
    try:
        subprocess.run("apt-get -qq -y install fonts-ipafont-gothic",
                       shell=True, check=True, timeout=300)
        fm._load_fontmanager(try_read_cache=False)
        matplotlib.rcParams["font.family"] = "IPAGothic"
        return True
    except Exception:
        return False


JP = _setup_japanese_font()


def L(ja, en):
    """日本語フォントが使えれば ja、駄目なら en を返す（図のラベル用）。"""
    return ja if JP else en


matplotlib.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "figure.dpi": 110, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.unicode_minus": False,
})

# 講義ノートの図と同じ色
C = {"blue": "#1f4e79", "red": "#c0392b", "green": "#1e8449",
     "orange": "#d68910", "purple": "#6a4c93", "gray": "#7f8c8d"}

print("日本語フォント:", "有効" if JP else "無効（図のラベルは英語になる）")
print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)

<a id="sec71"></a>
## 7.1 特徴写像とカーネルトリック

§7.1 の出発点を再現する。$\mathcal{X}=\mathbb{R}^2$ の同心円データは直線では分離できないが、
$$\phi(\boldsymbol{x})=(x_1^2,\ \sqrt2\,x_1x_2,\ x_2^2)^\top\in\mathbb{R}^3$$
で持ち上げると平面で分離できる。しかも講義ノート §7.1 の計算
$\langle\phi(\boldsymbol{x}),\phi(\boldsymbol{y})\rangle=(\boldsymbol{x}^\top\boldsymbol{y})^2$ により、
$\phi$ を作らずに 2 次元の内積を一回二乗するだけで済む。まずこの恒等式を数値で確かめる。

In [ ]:
from sklearn.datasets import make_circles


def phi2(X):
    """φ(x) = (x1², √2·x1x2, x2²)。X は 2×n（列がサンプル）、戻り値は 3×n。"""
    return np.vstack([X[0]**2, np.sqrt(2) * X[0] * X[1], X[1]**2])


x = np.array([[1.0], [2.0]])          # 講義ノート §7.1 と同じ 2 点（d×1）
y = np.array([[3.0], [-1.0]])
print("φ(x) =", np.round(phi2(x).ravel(), 4))
print("φ(y) =", np.round(phi2(y).ravel(), 4))
print("<φ(x),φ(y)> =", float(phi2(x).ravel() @ phi2(y).ravel()))
print("(xᵀy)²       =", float((x.ravel() @ y.ravel())**2))

# 200 点でまとめて確認（d×n 規約：列がサンプル）
rng = np.random.default_rng(0)
A, B = rng.normal(size=(2, 200)), rng.normal(size=(2, 200))
print("200点での最大誤差 =", np.abs(phi2(A).T @ phi2(B) - (A.T @ B)**2).max())

In [ ]:
Xnd, lab = make_circles(n_samples=200, factor=0.35, noise=0.06, random_state=0)
X = Xnd.T                     # sklearn は n×d を返すので転置して d×n にする
F = phi2(X)                   # 3×n の特徴行列

fig = plt.figure(figsize=(9.5, 4.0))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
for c, col in [(0, C["blue"]), (1, C["red"])]:
    m = lab == c
    ax1.scatter(X[0, m], X[1, m], s=14, c=col, label=f"class {c}")
    ax2.scatter(F[0, m], F[1, m], F[2, m], s=12, c=col)
ax1.set_aspect("equal"); ax1.legend(loc="upper right", fontsize=9)
ax1.set_xlabel("$x_1$"); ax1.set_ylabel("$x_2$")
ax1.set_title(L("入力空間 $\\mathbb{R}^2$", "input space $\\mathbb{R}^2$"))

gx, gy = np.meshgrid(np.linspace(0, 1.1, 2), np.linspace(-0.8, 0.8, 2))
ax2.plot_surface(gx, gy, 0.42 - gx, alpha=0.25, color=C["gray"])
ax2.set_xlabel("$z_1=x_1^2$"); ax2.set_ylabel("$z_2=\\sqrt{2}x_1x_2$")
ax2.set_zlabel("$z_3=x_2^2$")
ax2.set_title(L("特徴空間 $\\mathbb{R}^3$ と分離平面",
                "feature space $\\mathbb{R}^3$ and separating plane"))
ax2.view_init(elev=18, azim=-62)
fig.tight_layout(); plt.show()

$\langle\phi(\boldsymbol{x}),\phi(\boldsymbol{y})\rangle=0.9999999999999978$ と $(\boldsymbol{x}^\top\boldsymbol{y})^2=1$ は
倍精度の丸めの範囲で一致し、200 点をまとめて比べても最大誤差は $2.8\times10^{-14}$ である。
右図で $z_1+z_3=x_1^2+x_2^2=\|\boldsymbol{x}\|^2$、すなわち**原点からの距離の二乗が特徴空間では線形な量**に
なっていることが見て取れる。灰色の平面 $z_1+z_3=0.42$ が二つのクラスを分離し、
入力空間に引き戻すと半径 $\sqrt{0.42}$ の円になる。

<a id="sec72"></a>
## 7.2 正定値カーネルと Gram 行列

§7.6 の定義（正定値カーネル）は、任意の有限個の点に対する Gram 行列
$\boldsymbol{K}=(k(x_i,x_j))_{ij}$ が半正定値、すなわち固有値がすべて非負であることを要求する。
代表的なカーネルについて最小固有値を実際に計算し、
**シグモイドカーネル $\tanh(\boldsymbol{x}^\top\boldsymbol{y}+1)$ だけが正定値でない**ことを確かめる。

In [ ]:
def sqdist(A, B):
    """A: d×n, B: d×m -> n×m の二乗距離行列。"""
    return (A * A).sum(0)[:, None] + (B * B).sum(0)[None, :] - 2 * A.T @ B


def l1dist(A, B):
    return np.abs(A[:, :, None] - B[:, None, :]).sum(0)


k_lin = lambda A, B: A.T @ B
k_poly = lambda A, B, c=1.0, p=3: (A.T @ B + c)**p
k_gauss = lambda A, B, s=1.0: np.exp(-sqdist(A, B) / (2 * s**2))
k_lap = lambda A, B, s=1.0: np.exp(-l1dist(A, B) / s)
k_sigm = lambda A, B, a=1.0, b=1.0: np.tanh(a * (A.T @ B) + b)
k_per = lambda A, B, p=1.0, s=1.0: np.exp(          # 各座標の周期カーネルの積
    -2 * (np.sin(np.pi * (A[:, :, None] - B[:, None, :]) / p)**2).sum(0) / s**2)

rng = np.random.default_rng(0)
X = rng.normal(size=(3, 200))          # d=3, n=200（列がサンプル）
mineig = lambda K: np.linalg.eigvalsh((K + K.T) / 2).min()

for name, K in [("線形", k_lin(X, X)), ("多項式 c=1,p=3", k_poly(X, X)),
                ("ガウス σ=1", k_gauss(X, X)), ("ラプラス σ=1", k_lap(X, X)),
                ("周期 p=1,σ=1", k_per(X, X)), ("シグモイド", k_sigm(X, X))]:
    print(f"{name:16s} min eig = {mineig(K): .4e}   rank = "
          f"{np.linalg.matrix_rank(K, tol=1e-10 * np.abs(K).max())}")

線形カーネルの最小固有値 $-6.4\times10^{-14}$ は理論値 $0$ の丸め誤差で、階数は $d=3$ で頭打ちになる。
多項式（$c=1,p=3$）の階数 $20$ は特徴空間の次元 $\binom{d+p}{p}=\binom63=20$ に一致する。
ガウス（$1.0\times10^{-8}$）・ラプラス（$6.7\times10^{-2}$）・周期（$7.4\times10^{-5}$）は
いずれも最小固有値が正で階数は $n=200$、すなわち特徴空間は無限次元である。
シグモイドだけが $-20.04$ という大きな負の固有値をもち、$\tanh$ が「類似度らしく見える」だけで
正定値カーネルではないことが分かる（対応する RKHS が存在しない）。

次に、カーネルの「形」$k(x,0)$ を描く。§7.6 の図（代表的なカーネルの断面）に対応する。

In [ ]:
t = np.linspace(-3, 3, 601)[None, :]
zero = np.zeros((1, 1))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for s, ls in [(0.3, ":"), (1.0, "-"), (2.0, "--")]:
    axes[0].plot(t[0], k_gauss(t, zero, s).ravel(), ls, color=C["blue"],
                 label=L(f"ガウス σ={s}", f"Gaussian σ={s}"))
axes[0].plot(t[0], k_lap(t, zero, 1.0).ravel(), "-", color=C["red"],
             label=L("ラプラス σ=1", "Laplacian σ=1"))
axes[0].plot(t[0], k_per(t, zero, 1.5, 1.0).ravel(), "-", color=C["green"],
             label=L("周期 p=1.5, σ=1", "periodic p=1.5, σ=1"))
axes[0].set_title(L("平行移動不変なカーネル $k(x,0)$",
                    "translation-invariant kernels $k(x,0)$"))

for p, ls in [(1, ":"), (2, "-"), (3, "--")]:
    axes[1].plot(t[0], k_poly(t, np.ones((1, 1)), 1.0, p).ravel() / 2**p, ls,
                 color=C["purple"], label=L(f"多項式 p={p}（$2^{{-p}}$倍）",
                                            f"polynomial p={p} (scaled)"))
axes[1].set_title(L("多項式カーネルの断面 $k(x,1)=(x+1)^p$",
                    "polynomial kernels $k(x,1)$"))
for ax in axes:
    ax.set_xlabel("$x$"); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

左図で、ガウスは原点で滑らか（微分可能）だがラプラスは尖っている。
この違いがそのまま RKHS に属する関数の滑らかさに翻訳される（§7.7 の Fourier 表示）。
帯域幅 $\sigma$ を小さくすると山が細くなり、$k(x_i,x_j)\approx0$（$i\ne j$）で Gram 行列は
単位行列に近づく。大きくすると全成分が $1$ に近づき階数が落ちる。
この両極端の間に「ちょうどよい」帯域幅がある、というのが実務上の論点である（第8章 §8.4）。

<a id="sec73"></a>
## 7.3 カーネルの演算則

§7.6 の定理「カーネルの演算則」は、正定値カーネルが
和・正数倍・**各点ごとの積（Schur 積）**・各点収束極限・$f(x)k(x,y)f(y)$・$\exp$ で閉じることを主張する。
講義ノートのコードリスト（カーネルの演算則の数値的検証）を出発点に、
(1) 演算で正定値性が保たれること、(2) **行列積では保たれない**こと、
(3) ガウスカーネルを演算則どおりに組み立てると直接計算と一致すること、を確かめる。

In [ ]:
sigma = 1.0
K1, K2 = k_lin(X, X), k_gauss(X, X, sigma)      # 線形とガウス（ともに n×n）
ops = {"k1 (線形)": K1, "k2 (ガウス)": K2, "k1+k2": K1 + K2, "3*k2": 3 * K2,
       "k1*k2 (Schur積)": K1 * K2, "exp(k1)": np.exp(K1),
       "f·k1·f": np.exp(-(X**2).sum(0))[:, None] * K1 * np.exp(-(X**2).sum(0))[None, :]}
for name, K in ops.items():
    print(f"{name:16s} min eig = {mineig(K): .3e}")

# 命題「ガウスカーネルは正定値」の構成: k(x,y) = f(x)·exp(xᵀy/σ²)·f(y)
f = np.exp(-(X**2).sum(0) / (2 * sigma**2))
built = f[:, None] * np.exp(K1 / sigma**2) * f[None, :]
print("\n構成と直接計算の最大差 =", np.abs(built - K2).max())

In [ ]:
# Schur 積は半正定値を保つが、行列積は対称性すら保たない（§7.6 の例）
A = np.array([[2.0, 1.0], [1.0, 2.0]])
Bm = np.array([[1.0, 0.0], [0.0, 0.0]])
print("eig(A)      =", np.linalg.eigvalsh(A))
print("eig(B)      =", np.linalg.eigvalsh(Bm))
print("eig(A∘B)    =", np.linalg.eigvalsh(A * Bm), "  ← Schur 積：半正定値")
print("A@B         =", np.round(A @ Bm, 3).tolist(), "  ← 対称でない")
print("B@A         =", np.round(Bm @ A, 3).tolist(), "  ← 対称でなく、A@B とも一致しない")

# 演算で「保たれない」例：正定値でないシグモイドを足すと壊れる
Ks = k_sigm(X, X)
print("\nmin eig(ガウス + シグモイド) =", f"{mineig(K2 + Ks): .3e}",
      "  ← 片方が正定値でなければ和も一般に正定値でない")

すべての演算で最小固有値は $0$ 以上（負に見えるのは $10^{-13}$ 台の丸め誤差）であり、
定理の主張どおり正定値性が保たれている。
構成 $k(\boldsymbol{x},\boldsymbol{y})=f(\boldsymbol{x})\exp(\boldsymbol{x}^\top\boldsymbol{y}/\sigma^2)f(\boldsymbol{y})$ と
$\exp(-\|\boldsymbol{x}-\boldsymbol{y}\|^2/2\sigma^2)$ の直接計算は $10^{-16}$ の水準で一致する。

一方、$\boldsymbol{A}\boldsymbol{B}$ と $\boldsymbol{B}\boldsymbol{A}$ はどちらも対称でなく互いにも異なる。
**カーネルの積は行列積ではなく成分ごとの積である**ことを取り違えないように。
最後の行は「正定値でないものを混ぜると壊れる」ことの確認で、演算則の仮定
（$k_1,k_2$ がともに正定値）が本質的であることを示している。

<a id="sec74"></a>
## 7.4 ガウスカーネルの明示的特徴写像

§7.8 の命題「1 次元ガウスカーネルの明示的特徴写像」は
$$\phi(x)=\Bigl(e^{-x^2/2\sigma^2}\tfrac{1}{\sqrt{m!}}(x/\sigma)^m\Bigr)_{m=0,1,2,\dots}\in\ell^2$$
が $k(x,y)=\exp(-(x-y)^2/2\sigma^2)$ の特徴写像であることを主張する。
無限次元だが、第 $N$ 項で打ち切ると何項で実用上一致するかを見る。

In [ ]:
from scipy.special import factorial

def phi_series(xs, N, s=1.0):
    """打ち切った明示的特徴写像。xs: 長さ n の 1 次元配列 -> n×(N+1)。"""
    m = np.arange(N + 1)
    return (np.exp(-xs**2 / (2 * s**2))[:, None]
            * (xs[:, None] / s)**m / np.sqrt(factorial(m)))


pts = np.linspace(-2, 2, 5)                     # 講義ノートと同じ 5 点、σ=1
Kex = np.exp(-(pts[:, None] - pts[None, :])**2 / 2)
errs = []
for N in [2, 5, 10, 20, 30]:
    P = phi_series(pts, N)
    e = np.abs(P @ P.T - Kex).max()
    errs.append((N, e))
    print(f"N={N:3d}  最大誤差 = {e:.3e}")

$[-2,2]$ の 5 点で評価すると、最大誤差は $N=5$ で $2.1\times10^{-1}$、$N=10$ で $2.8\times10^{-3}$、
$N=20$ で $1.9\times10^{-9}$、$N=30$ で $1.1\times10^{-16}$（倍精度の丸め誤差）まで落ちる。
無限次元の内積が有限項で実用上完全に再現され、しかも $k$ 自体は $O(1)$ で計算できる。
これがカーネルトリックの威力である。

<a id="sec75"></a>
## 7.5 再生性と RKHS の元

§7.7 の定理「再生核の存在」は $f(x)=\langle f,k(\cdot,x)\rangle_{\mathcal{H}}$（再生性）を、
Moore–Aronszajn の定理は $\mathcal{H}_0=\mathrm{span}\{k(\cdot,x)\}$ の完備化が RKHS になることを述べた。
ここでは有限個の中心をもつ $f=\sum_i\alpha_ik(\cdot,x_i)\in\mathcal{H}_0$ を実際に描き、
$\|f\|_{\mathcal{H}}^2=\boldsymbol{\alpha}^\top\boldsymbol{K}\boldsymbol{\alpha}$ と
$|f(x)|\le\|f\|_{\mathcal{H}}\sqrt{k(x,x)}$ を数値で確かめる。

In [ ]:
ctr = np.array([[-2.0, -1.0, 0.0, 1.0, 2.0]])      # 中心 x_i（1×5、列がサンプル）
al = np.array([1.0, -1.5, 2.0, -1.0, 0.5])          # 係数 α
grid = np.linspace(-4, 4, 801)[None, :]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, s in zip(axes, [0.4, 1.2]):
    K = k_gauss(ctr, ctr, s)
    fval = (k_gauss(grid, ctr, s) @ al)
    nrm = np.sqrt(al @ K @ al)
    for i in range(ctr.shape[1]):                   # 各 α_i k(·,x_i)
        ax.plot(grid[0], al[i] * k_gauss(grid, ctr[:, i:i+1], s).ravel(),
                color=C["gray"], lw=0.8, alpha=0.7)
    ax.plot(grid[0], fval, color=C["blue"], lw=2, label="$f=\\sum_i\\alpha_ik(\\cdot,x_i)$")
    ax.axhline(0, color="k", lw=0.6)
    ax.scatter(ctr[0], np.zeros(5), c=C["red"], zorder=5, s=18)
    ax.set_title(L(f"σ={s}:  ‖f‖_H = {nrm:.3f}", f"σ={s}:  ||f||_H = {nrm:.3f}"))
    ax.set_xlabel("$x$"); ax.legend(fontsize=9)
    print(f"σ={s}: ‖f‖²_H = αᵀKα = {al @ K @ al:.4f}, "
          f"max|f| = {np.abs(fval).max():.4f}, "
          f"上界 ‖f‖_H√k(x,x) = {nrm:.4f}")
fig.tight_layout(); plt.show()

細い灰色の曲線が $\alpha_ik(\cdot,x_i)$、太い青が和 $f$ である。
**RKHS の元はカーネルの山の重ね合わせ**であり、帯域幅 $\sigma$ が小さいほど山が細く $f$ は激しく振れ、
大きいほど滑らかになる。$\sigma=0.4$ では $\|f\|_{\mathcal{H}}^2=7.8849$（$\|f\|_{\mathcal{H}}=2.808$）、
$\sigma=1.2$ では $0.7012$（$0.837$）で、同じ $\boldsymbol{\alpha}$ でも滑らかな方がノルムが小さい。
どちらも $\max|f|$（それぞれ $1.891$ と $0.612$）が上界
$\|f\|_{\mathcal{H}}\sqrt{k(x,x)}=\|f\|_{\mathcal{H}}$（ガウスでは $k(x,x)=1$）を下回っている。

次に再生性そのものを確かめる。$\mathcal{H}$ での内積は無限次元の操作だが、
Mercer 展開（§7.9）で得られる有限次元の特徴 $\phi_r(x)=(\sqrt{\lambda_i}e_i(x))_{i\le r}$ を使えば
**$\mathbb{R}^r$ の普通の内積**として計算できる。$r$ を増やすと $\langle f,k(\cdot,x)\rangle$ が
$f(x)$ に収束することを見る。

In [ ]:
# [-4,4] 上で積分作用素を離散化し、Mercer 特徴 φ_r(x) = (√λ_i e_i(x))_{i≤r} を作る
s = 0.8
mg = 600
gp = np.linspace(-4, 4, mg)
w = 8.0 / mg
Kg = k_gauss(gp[None, :], gp[None, :], s)
vals, vecs = np.linalg.eigh(w * Kg)
lam = vals[::-1]
ef = vecs[:, ::-1] / np.sqrt(w)                    # ∫e_i² dμ ≈ 1 になる正規化

Kcc = k_gauss(ctr, ctr, s)
alpha = np.linalg.solve(Kcc + 1e-12 * np.eye(5), np.array([0.5, -1.0, 1.5, -0.5, 1.0]))
xq = np.array([-1.3, 0.0, 0.7, 2.1])               # 再生性を確かめる点
idx_c = [np.argmin(np.abs(gp - c)) for c in ctr[0]]
idx_q = [np.argmin(np.abs(gp - q)) for q in xq]
ctr_g, xq_g = gp[idx_c][None, :], gp[idx_q][None, :]   # 格子点に丸めた中心・評価点
f_exact = k_gauss(xq_g, ctr_g, s) @ alpha              # f(x) = Σ α_i k(x_i,x)

print(" r    max |<f,k(·,x)>_r - f(x)|")
for r in [2, 5, 10, 20, 40]:
    Phi = ef[:, :r] * np.sqrt(lam[:r])             # 格子点上の φ_r
    cf = Phi[idx_c].T @ alpha                      # f の係数ベクトル（r 次元）
    approx = Phi[idx_q] @ cf                       # <f, φ_r(x)> = 内積
    print(f"{r:3d}    {np.abs(approx - f_exact).max():.3e}")

$r=2$ では $1.30$ とまったく合わないが、$r=10$ で $5.8\times10^{-2}$、$r=20$ で $2.2\times10^{-8}$、
$r=40$ で $1.0\times10^{-14}$ まで下がる。
すなわち**再生性 $f(x)=\langle f,k(\cdot,x)\rangle$ は有限次元の近似で確かに成り立っている**。
無限次元の内積は、Mercer 座標で見れば $\mathbb{R}^r$ の内積の極限にすぎない。

<a id="sec76"></a>
## 7.6 Mercer 展開

§7.9 の Mercer の定理は、コンパクト集合上の連続な正定値カーネルが
$k(x,y)=\sum_i\lambda_ie_i(x)e_i(y)$ と一様絶対収束することを主張する。
$\mathcal{X}=[-1,1]$、$\mu$ を一様測度、$\sigma=0.3$ のガウスカーネルとし、
講義ノートのコードリスト（Mercer 固有関数の数値計算）どおり
$m=400$ 点の格子で積分作用素を行列 $w\boldsymbol{K}$（$w=2/m$）に離散化して固有分解する。

In [ ]:
m, sg = 400, 0.3
gr = np.linspace(-1.0, 1.0, m)
wt = 2.0 / m
Km = np.exp(-(gr[:, None] - gr[None, :])**2 / (2 * sg**2))
va, ve = np.linalg.eigh(wt * Km)
lm, ee = va[::-1], ve[:, ::-1] / np.sqrt(wt)
ee = ee * np.sign(ee[m // 2 + 7])                  # 符号の向きを揃える

print("上位6固有値:", np.round(lm[:6], 4))
print("∫e_1² dμ =", round(float((ee[:, 0]**2).sum() * wt), 6),
      "  ∫e_1e_2 dμ =", f"{float((ee[:, 0] * ee[:, 1]).sum() * wt):.2e}")
tail = np.array([lm[r:].sum() for r in range(1, 25)])
err = np.array([np.abs((ee[:, :r] * lm[:r]) @ ee[:, :r].T - Km).max()
                for r in range(1, 25)])
for r in [2, 6, 12, 20]:
    print(f"r={r:2d}  打ち切り誤差 = {err[r-1]:.3e}   固有値の裾 Σ_(i>r)λ_i = {tail[r-1]:.3e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.5))
for i, col in enumerate([C["blue"], C["red"], C["green"], C["orange"]]):
    axes[0].plot(gr, ee[:, i], color=col, label=f"$e_{i+1}$")
axes[0].set_xlabel("$x$"); axes[0].legend(fontsize=8, ncol=2)
axes[0].set_title(L("Mercer 固有関数（σ=0.3）", "Mercer eigenfunctions (σ=0.3)"))

for sig2, col in [(0.1, C["orange"]), (0.3, C["blue"]), (1.0, C["green"])]:
    Ks2 = np.exp(-(gr[:, None] - gr[None, :])**2 / (2 * sig2**2))
    l2 = np.linalg.eigvalsh(wt * Ks2)[::-1]
    axes[1].semilogy(np.arange(1, 26), np.maximum(l2[:25], 1e-18), "o-", ms=3,
                     color=col, label=f"σ={sig2}")
axes[1].set_xlabel("$i$"); axes[1].set_ylabel("$\\lambda_i$")
axes[1].legend(fontsize=8)
axes[1].set_title(L("固有値の減衰", "eigenvalue decay"))

axes[2].semilogy(np.arange(1, 25), err, "o-", ms=3, color=C["red"],
                 label=L("打ち切り誤差", "truncation error"))
axes[2].semilogy(np.arange(1, 25), np.maximum(tail, 1e-18), "s--", ms=3,
                 color=C["gray"], label=L("固有値の裾 $\\sum_{i>r}\\lambda_i$",
                                          "tail $\\sum_{i>r}\\lambda_i$"))
axes[2].set_xlabel("$r$"); axes[2].legend(fontsize=8)
axes[2].set_title(L("打ち切り誤差は裾で決まる", "error follows the tail"))
fig.tight_layout(); plt.show()

上位 6 固有値は $0.6924,\ 0.5452,\ 0.3675,\ 0.2131,\ 0.1072,\ 0.0472$ と急速に減衰し、
固有関数は番号が上がるほど節（ゼロ点）が増えて振動が激しくなる（左図）。
$\int e_1^2\,d\mu=1.0$、$\int e_1e_2\,d\mu=7.1\times10^{-17}$ で正規直交性も確認できる。

中央の図で、$\sigma$ が大きいほど固有値の減衰が速い。
$\|f\|_{\mathcal{H}}^2=\sum_ic_i^2/\lambda_i$（§7.9 の系「Mercer 特徴写像」）だから、
$\lambda_i$ が速く減衰するほど高次成分の罰則が厳しく、RKHS は狭く（関数は滑らかに）なる。
これは §7.7 の Fourier 表示 $\|f\|^2\propto\int|\hat f|^2e^{\sigma^2\omega^2/2}d\omega$ と整合する。

右図が本節の要点である。打ち切り誤差 $\max_{x,y}|k-\sum_{i\le r}\lambda_ie_ie_i|$ は
$r=2$ で $0.803$、$r=6$ で $8.60\times10^{-2}$、$r=12$ で $6.46\times10^{-5}$、
$r=20$ で $4.22\times10^{-11}$ と減り、固有値の裾 $\sum_{i>r}\lambda_i$
（同じ $r$ で $0.762$、$2.75\times10^{-2}$、$9.79\times10^{-6}$、$5.31\times10^{-12}$）と
片対数で平行に落ちている（誤差は裾の $3$–$8$ 倍に収まっている）。
**打ち切り誤差は固有値の裾で決まる**（$|k-k_r|\le\sum_{i>r}\lambda_i$ が Cauchy–Schwarz から出る）。

<a id="sec77"></a>
## 7.7 MMD による二標本の比較

RKHS の道具は点だけでなく**確率分布**にも使える。
$\mu_P=\mathbb{E}_{X\sim P}[k(\cdot,X)]$（カーネル平均埋め込み）と
$\mathrm{MMD}(P,Q)=\|\mu_P-\mu_Q\|_{\mathcal{H}}$ は第8章 §8.6 の主題だが、
「$\mathcal{H}$ の幾何をそのまま分布に適用する」例として本章の締めに置く。
不偏推定量（$U$ 統計量）は
$$\widehat{\mathrm{MMD}}_u^2=\tfrac{1}{n(n-1)}\sum_{i\ne j}k(x_i,x_j)
+\tfrac{1}{m(m-1)}\sum_{i\ne j}k(y_i,y_j)-\tfrac{2}{nm}\sum_{i,j}k(x_i,y_j)$$
である。$P=Q$ のとき負の値もとりうる点に注意する。

In [ ]:
def mmd2_u(Kxx, Kyy, Kxy):
    """Gram 行列から U 統計量版の MMD²（対角を除く）。"""
    n, m = Kxx.shape[0], Kyy.shape[0]
    return ((Kxx.sum() - np.trace(Kxx)) / (n * (n - 1))
            + (Kyy.sum() - np.trace(Kyy)) / (m * (m - 1))
            - 2 * Kxy.mean())


def mmd_perm_test(Xs, Ys, kfun, B=300, seed=0):
    """置換検定。Xs, Ys は 1×n / 1×m（列がサンプル）。"""
    n, m = Xs.shape[1], Ys.shape[1]
    Z = np.hstack([Xs, Ys])
    G = kfun(Z, Z)
    obs = mmd2_u(G[:n, :n], G[n:, n:], G[:n, n:])
    rg = np.random.default_rng(seed)
    null = np.empty(B)
    for b in range(B):
        p = rg.permutation(n + m)
        Gp = G[np.ix_(p, p)]
        null[b] = mmd2_u(Gp[:n, :n], Gp[n:, n:], Gp[:n, n:])
    return obs, null, (1 + (null >= obs).sum()) / (B + 1)


rg = np.random.default_rng(0)
n = m = 200
P = rg.normal(0.0, 1.0, (1, n))
Q_same = rg.normal(0.0, 1.0, (1, m))               # 同分布
Q_shift = rg.normal(0.5, 1.0, (1, m))              # 平均だけずらす
Q_scale = rg.normal(0.0, 2.0, (1, m))              # 平均は同じで分散が違う

kg = lambda A, B: k_gauss(A, B, 1.0)
res = {}
for name, Q in [("同分布 N(0,1)", Q_same), ("平均シフト N(0.5,1)", Q_shift),
                ("分散違い N(0,2²)", Q_scale)]:
    obs, null, p = mmd_perm_test(P, Q, kg)
    res[name] = (obs, null, p)
    lin = mmd2_u(k_lin(P, P), k_lin(Q, Q), k_lin(P, Q))
    print(f"{name:18s} ガウス MMD²_u = {obs: .5f}  p = {p:.4f}   "
          f"線形カーネル MMD²_u = {lin: .5f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.3), sharey=True)
for ax, (name, (obs, null, p)) in zip(axes, res.items()):
    ax.hist(null, bins=30, color=C["gray"], alpha=0.75,
            label=L("置換分布（帰無仮説）", "permutation null"))
    ax.axvline(obs, color=C["red"], lw=2,
               label=L("観測値", "observed"))
    ax.set_title(L(f"{name}   p={p:.3f}", f"p={p:.3f}"), fontsize=10)
    ax.set_xlabel("$\\widehat{\\mathrm{MMD}}^2_u$")
axes[0].set_ylabel(L("頻度", "count")); axes[0].legend(fontsize=8)
fig.tight_layout(); plt.show()

同分布では $\widehat{\mathrm{MMD}}^2_u=0.00415$ で置換検定の $p=0.146$、すなわち棄却されない。
平均シフトでは $0.04245$、$p=0.003$（$B=300$ 回の置換で一度も超えられない最小値）と明確に棄却される。
平均は同じで分散だけが違う $\mathcal{N}(0,2^2)$ でも、ガウスカーネルなら $0.10794$、$p=0.003$ で検出できる。
一方**線形カーネルの $\widehat{\mathrm{MMD}}^2_u$ は、平均シフトでは $0.230$ と大きいのに
分散違いでは $-0.0173$ と $0$ 付近にとどまる**。
線形カーネルの平均埋め込みは $\mu_P=\mathbb{E}[X]$ なので平均しか見ておらず、
特性的（characteristic）でない——分布を区別できない——ことがここに現れている。

<a id="ex"></a>
## 演習

**問 1（正定値性の判定）**
$k(x,y)=\tanh(xy+1)$ を $x_1=-2,\ x_2=0,\ x_3=2$ の 3 点で評価して Gram 行列を作り、
固有値を求めて正定値かどうか判定せよ。トレースと固有値の和が一致することも確かめよ。

**問 2（帯域幅と数値的階数）**
$\boldsymbol{X}\in\mathbb{R}^{5\times60}$ を標準正規乱数（`default_rng(0)`）で作り、
ガウスカーネルの帯域幅 $\sigma$ を $1$ から $1000$ まで対数的に変えたときの
Gram 行列の数値的階数（固有値が $10^{-10}\max_i\lambda_i$ を超える個数）と
$\lambda_2/\lambda_1$ を求めよ。$\sigma\to\infty$ で階数が $1$ に「落ちきらない」のはなぜか。

**問 3（最小ノルム補間）**
$\sigma=1$ のガウスカーネル、$x_1=0,x_2=1,x_3=2$ とする。
$f(0)=1,\ f(1)=0,\ f(2)=1$ を満たす $f\in\mathcal{H}$ のうち $\|f\|_{\mathcal{H}}$ が最小のものは
$f=\sum_i\alpha_ik(\cdot,x_i)$ の形で、$\boldsymbol{\alpha}=\boldsymbol{K}^{-1}\boldsymbol{y}$ で与えられる（表現定理の原型）。
$\boldsymbol{\alpha}$ と $\|f\|_{\mathcal{H}}^2=\boldsymbol{y}^\top\boldsymbol{K}^{-1}\boldsymbol{y}$ を求めよ。
また $\boldsymbol{\alpha}$ に摂動 $t(1,-1,1)^\top$ を加えると補間条件が崩れること
（$\boldsymbol{K}$ は正則なので補間解は一意）と、同時にノルムも増えることを確かめよ。

In [ ]:
# 問 1
xs1 = np.array([[-2.0, 0.0, 2.0]])
K1_ex = np.tanh(xs1.T @ xs1 + 1.0)
# TODO: K1_ex の固有値を求め、最小固有値の符号から正定値かどうか判定する
# TODO: np.trace(K1_ex) と固有値の和を比べる

In [ ]:
# 問 2
Xq = np.random.default_rng(0).normal(size=(5, 60))
for sg2 in [1.0, 10.0, 100.0, 1000.0]:
    Kq = k_gauss(Xq, Xq, sg2)
    ev = np.linalg.eigvalsh(Kq)[::-1]
    # TODO: 数値的階数（ev > 1e-10*ev[0] の個数）と ev[1]/ev[0] を出力する
    pass

In [ ]:
# 問 3
xs3 = np.array([[0.0, 1.0, 2.0]])
y3 = np.array([1.0, 0.0, 1.0])
K3 = k_gauss(xs3, xs3, 1.0)
# TODO: alpha = K3⁻¹y3 を解き、‖f‖² = αᵀK3α = y3ᵀK3⁻¹y3 を求める
# TODO: alpha に K3 の核でない適当な摂動 δ を足すと補間条件が崩れることを確認する
#       （K3 は正則なので、補間条件 K3α=y を満たす α は一意である）

<a id="sol"></a>
## 演習の解答

In [ ]:
# --- 問 1 ---
ev1 = np.linalg.eigvalsh(K1_ex)
print("K =\n", np.round(K1_ex, 4))
print("固有値 =", np.round(ev1, 4))
print("正定値か:", bool(ev1.min() >= -1e-12),
      " / trace =", round(float(np.trace(K1_ex)), 4),
      "= Σλ =", round(float(ev1.sum()), 4))

In [ ]:
# --- 問 2 ---
print(" σ      数値的階数   λ2/λ1")
for sg2 in [1.0, 10.0, 100.0, 1000.0]:
    ev = np.linalg.eigvalsh(k_gauss(Xq, Xq, sg2))[::-1]
    print(f"{sg2:7.0f}   {int((ev > 1e-10 * ev[0]).sum()):6d}      {ev[1]/ev[0]:.2e}")

$\sigma=1,10,100,1000$ で数値的階数は $60,\ 59,\ 21,\ 6$、$\lambda_2/\lambda_1$ は
$5.66\times10^{-1},\ 1.61\times10^{-2},\ 1.67\times10^{-4},\ 1.67\times10^{-6}$ である。
$K_{ij}=1-\|\boldsymbol{x}_i-\boldsymbol{x}_j\|^2/(2\sigma^2)+O(\sigma^{-4})$ と展開すると第二項が $\sigma^{-2}$ の
オーダーで残るので、$\lambda_2/\lambda_1$ は $\sigma^{-2}$ でしか減らない（表の比が $\sigma$ を
10 倍するごとに約 $1/100$ になっている）。
極限では階数 $1$ だが、有限の $\sigma$ では相対閾値 $10^{-10}$ の下で階数は $1$ に落ちきらない。
**「極限で階数 1」と「有限の $\sigma$ で数値的階数 1」を混同しない**こと。

In [ ]:
# --- 問 3 ---
alpha3 = np.linalg.solve(K3, y3)
print("K =\n", np.round(K3, 4))
print("α =", np.round(alpha3, 4))
print("‖f‖² = αᵀKα =", round(float(alpha3 @ K3 @ alpha3), 4),
      " = yᵀK⁻¹y =", round(float(y3 @ alpha3), 4))
print("補間の確認 Kα =", np.round(K3 @ alpha3, 6))

# 補間条件を保つ摂動は存在しない（K は正則）。ノルムだけ増える例を見る
for t in [0.0, 0.3, 1.0]:
    a = alpha3 + t * np.array([1.0, -1.0, 1.0])
    print(f"t={t}: f(x_i) = {np.round(K3 @ a, 4)},  ‖f‖² = {a @ K3 @ a:.4f}")

$\boldsymbol{\alpha}=(2.5027,-3.0359,2.5027)^\top$、$\|f\|_{\mathcal{H}}^2=5.0053$ である
（$\boldsymbol{K}\boldsymbol{\alpha}=(1,0,1)^\top$ で補間条件を満たす）。
$\boldsymbol{K}$ が正則なので補間条件 $\boldsymbol{K}\boldsymbol{\alpha}=\boldsymbol{y}$ を満たす $\boldsymbol{\alpha}$ は一意であり、
摂動 $t(1,-1,1)^\top$ を加えると $t=0.3$ で $f(x_i)=(1.1586,0.0639,1.1586)$、
$t=1$ で $(1.5288,0.2131,1.5288)$ と補間条件が崩れ、
同時にノルムの二乗も $5.0053\to6.2813\to9.8498$ と増える。
無限次元 $\mathcal{H}$ での最小ノルム補間が $n=3$ 変数の線形方程式に帰着している——
これが第8章の表現定理の原型である。